### Mini Project — PCA + Model Training

High-Dimensional Customer Data — Compress Then Predict

In [7]:
import numpy as np

np.random.seed(50)
n_samples = 400
n_features = 20   # many features, some of which are correlated

# Base signals
base1 = np.random.normal(0, 1, n_samples)
base2 = np.random.normal(0, 1, n_samples)

X = np.zeros((n_samples, n_features))
for i in range(n_features):
    if i < 10:
        X[:, i] = base1 * np.random.uniform(0.5, 1.5) + np.random.normal(0, 0.3, n_samples)
    else:
        X[:, i] = base2 * np.random.uniform(0.5, 1.5) + np.random.normal(0, 0.3, n_samples)

# Target depends on the underlying base signals
y = (base1 + base2 + np.random.normal(0, 0.5, n_samples) > 0).astype(int)

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Scaler = StandardScaler()

X_train_scaled = Scaler.fit_transform(X_train)
X_test_scaled = Scaler.transform(X_test)


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=0.95)
X_train_reduced = pca.fit_transform(X_train_scaled)
X_test_reduced = pca.transform(X_test_scaled)

print(f"Number of components to explain 95% variance: {pca.n_components_}")

Number of components to explain 95% variance: 6


In [10]:
import time
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

start_train = time.time()
model.fit(X_train_reduced, y_train)
end_train = time.time()
train_time = end_train - start_train

score = model.score(X_test_reduced, y_test)

print(f"Model score on test set: {score:.2%}")
print(f"Training time: {train_time:.2f} seconds")

Model score on test set: 87.50%
Training time: 0.01 seconds


In [11]:
# Without pcs

start_train = time.time()
model.fit(X_train_scaled, y_train)
end_train = time.time()
train_time = end_train - start_train

score = model.score(X_test_scaled, y_test)

print(f"Model score on test set without PCA: {score:.2%}")
print(f"Training time: {train_time:.2f} seconds")

Model score on test set without PCA: 92.50%
Training time: 0.00 seconds


In [12]:
from sklearn.naive_bayes import GaussianNB

model_NB= GaussianNB()

start_train = time.time()
model_NB.fit(X_train_reduced, y_train)
end_train = time.time()
train_time = end_train - start_train

score = model_NB.score(X_test_reduced, y_test)

print(f"Model score on test set: {score:.2%}")
print(f"Training time: {train_time:.2f} seconds")

Model score on test set: 91.25%
Training time: 0.00 seconds


In [13]:
# Without pcs

start_train = time.time()
model_NB.fit(X_train_scaled, y_train)
end_train = time.time()
train_time = end_train - start_train

score = model_NB.score(X_test_scaled, y_test)

print(f"Model score on test set without PCA: {score:.2%}")
print(f"Training time: {train_time:.2f} seconds")

Model score on test set without PCA: 88.75%
Training time: 0.00 seconds


In [21]:
from sklearn.model_selection import cross_val_score
scores_with_pca = cross_val_score(LogisticRegression(), X_train_reduced, y_train, cv=5)
scores_without_pca = cross_val_score(LogisticRegression(), X_train_scaled, y_train, cv=5)

print(f"Cross-validation scores with PCA: {scores_with_pca}")
print(f"Mean cross-validation score with PCA: {np.mean(scores_with_pca):.2%}")
print(f"Cross-validation scores without PCA: {scores_without_pca}")
print(f"Mean cross-validation score without PCA: {np.mean(scores_without_pca):.2%}")

Cross-validation scores with PCA: [0.90625  0.84375  0.90625  0.921875 0.953125]
Mean cross-validation score with PCA: 90.62%
Cross-validation scores without PCA: [0.90625  0.84375  0.921875 0.90625  0.953125]
Mean cross-validation score without PCA: 90.62%
